In [ ]:
import numpy as np
import gymnasium as gym

env = gym.make("CliffWalking-v1")
env.observation_space, env.action_space

In [ ]:
def train(method, episodes=1000, alpha=0.5, gamma=1.0, eps=0.1, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((48, 4))
    returns = []

    def act(s):
        if rng.random() < eps:
            return int(rng.integers(4))
        return int(np.argmax(Q[s]))

    for ep in range(episodes):
        s, _ = env.reset(seed=seed + ep)
        a, done, total = act(s), False, 0
        while not done:
            s2, r, term, trunc, _ = env.step(a)
            done = term or trunc
            a2 = act(s2)
            if method == "sarsa":
                target = r + gamma * Q[s2, a2] * (not term)
            else:
                target = r + gamma * Q[s2].max() * (not term)
            Q[s, a] += alpha * (target - Q[s, a])
            s, a, total = s2, a2, total + r
        returns.append(total)
    return Q, returns

In [ ]:
Q_sarsa, ret_sarsa = train("sarsa")
Q_q, ret_q = train("q")

In [ ]:
def path(Q):
    grid = np.full((4, 12), ".")
    grid[3, 1:11] = "C"
    s, _ = env.reset()
    for _ in range(50):
        r, c = divmod(s, 12)
        grid[r, c] = "o"
        s, _, term, _, _ = env.step(int(np.argmax(Q[s])))
        if term:
            break
    grid[3, 11] = "G"
    print("\n".join(" ".join(row) for row in grid))

path(Q_sarsa)
print()
path(Q_q)

In [ ]:
import matplotlib.pyplot as plt

smooth = lambda x: np.convolve(x, np.ones(50) / 50, mode="valid")
plt.plot(smooth(ret_sarsa), label="SARSA")
plt.plot(smooth(ret_q), label="Q-learning")
plt.ylim(-100, 0)
plt.xlabel("episode")
plt.ylabel("return")
plt.legend();

In [ ]:
import collections, copy, random
import torch
from torch import nn

torch.manual_seed(1); random.seed(1)
env = gym.make("CartPole-v1")
env.action_space.seed(1)
q = nn.Sequential(nn.Linear(4, 120), nn.ReLU(), nn.Linear(120, 84), nn.ReLU(), nn.Linear(84, 2))
target = copy.deepcopy(q)
opt = torch.optim.Adam(q.parameters(), lr=2.5e-4)
buffer = collections.deque(maxlen=10_000)

In [ ]:
TOTAL = 200_000
s, _ = env.reset(seed=1)
ep_len, lens = 0, []
for t in range(TOTAL):
    eps = max(0.05, 1 - t / (0.5 * TOTAL))
    if random.random() < eps:
        a = env.action_space.sample()
    else:
        a = int(q(torch.tensor(s)).argmax())
    s2, r, term, trunc, _ = env.step(a)
    buffer.append((s, a, r, s2, term))
    s, ep_len = s2, ep_len + 1
    if term or trunc:
        lens.append(ep_len)
        s, ep_len = env.reset()[0], 0
    if t >= 1000 and t % 10 == 0:
        S, A, R, S2, D = map(np.array, zip(*random.sample(buffer, 128)))
        with torch.no_grad():
            y = torch.tensor(R, dtype=torch.float32) + 0.99 * target(torch.tensor(S2)).max(1).values * (1 - torch.tensor(D, dtype=torch.float32))
        pred = q(torch.tensor(S)).gather(1, torch.tensor(A)[:, None]).squeeze(1)
        loss = nn.functional.mse_loss(pred, y)
        opt.zero_grad(); loss.backward(); opt.step()
    if t % 500 == 0:
        target.load_state_dict(q.state_dict())
    if t % (TOTAL // 10) == 0 and lens:
        print(t, np.mean(lens[-20:]))

In [ ]:
import mediapy

env = gym.make("CartPole-v1", render_mode="rgb_array")
s, _ = env.reset(seed=123)
frames = []
for t in range(500):
    frames.append(env.render())
    s, r, term, trunc, _ = env.step(int(q(torch.tensor(s)).argmax()))
    if term or trunc:
        break
print(len(frames), "步")
mediapy.show_video(frames, fps=50, width=480)